# Silver Table 14: regional flood exposure

## Purpose

Perform the governed MGB-to-region spatial intersection once in Silver. The notebook clips approved MGB flood-susceptibility polygons to safe official region boundaries, dissolves overlapping pieces, and calculates regional exposure area in one documented equal-area CRS. Future Gold `fact_region_flood_exposure` assigns serving keys only. It must not repeat code mapping, geometry parsing, clipping, area calculation, overlap handling, or boundary-version reconciliation.

This table measures hazard exposure only. It does not compare exposure with investment, and it does not judge whether flood-control spending is sufficient, fair, or effective.

## Grain

One official PSGC region × one active `APPROVED` standardized MGB susceptibility level × the selected MGB snapshot × the selected boundary snapshot × one deterministic flood-exposure run. The region set comes from `silver_psgc_place` and the level set comes from configuration. Neither count is hardcoded.

## Source authority

| Input | Role |
| --- | --- |
| `01-bronze.flood_susceptibility` | Selected MGB flood-area rows, raw susceptibility codes, and `geometry_json` |
| `02-silver.config_mgb_susceptibility_mapping` | The only authority for code-to-level and severity-rank standardization |
| `02-silver.silver_psgc_place` | The current official region set and PSGC lineage |
| `01-bronze.boundaries` | Region boundary geometry, boundary version, and boundary lineage |
| `01-bronze.load_log` and validation results | Compact upstream safety evidence |

Table 14 does not read DPWH projects, flood-control projects, Census Table C, population outputs, Table 13, or Gold. It never rereads R2 directly.

## 1. Confirm upstream source safety

**What:** require the newest terminal MGB and boundary loads to be `SUCCESS` or `SKIPPED_IDEMPOTENT`, require their latest Bronze validation runs to have no blocking failure, and require safe CONFIG and PSGC Silver validation evidence.

**Why:** the newest audit event wins, so an older success cannot hide a newer failed load. Only compact audit metadata and lineage columns are read here.

**Protects:** one selected MGB snapshot, one selected boundary snapshot, and one current official PSGC region set.

**Next proof:** the validator repeats these timeliness checks against the published lineage.

In [0]:
%sql
USE CATALOG `buildabida-capstone`;
CREATE SCHEMA IF NOT EXISTS `02-silver`;

-- newest terminal audit event per source, including failed or unfinished loads
CREATE OR REPLACE TEMPORARY VIEW latest_flood_exposure_source_load AS
SELECT * EXCEPT (terminal_rank)
FROM (
    SELECT *, ROW_NUMBER() OVER (
        PARTITION BY table_name
        ORDER BY COALESCE(completed_at, started_at) DESC, started_at DESC
    ) AS terminal_rank
    FROM `01-bronze`.load_log
    WHERE table_name IN ('flood_susceptibility', 'boundaries')
      AND status IN ('STARTED', 'SUCCESS', 'FAILED', 'SKIPPED_IDEMPOTENT')
)
WHERE terminal_rank = 1;

CREATE OR REPLACE TEMPORARY VIEW latest_flood_exposure_bronze_validation AS
SELECT * EXCEPT (validation_rank)
FROM (
    SELECT dq.*, DENSE_RANK() OVER (
        PARTITION BY dq.table_name ORDER BY dq.run_ts DESC, dq.run_id DESC
    ) AS validation_rank
    FROM `04-validation`.dq_results AS dq
    INNER JOIN latest_flood_exposure_source_load AS load
      ON dq.table_name = load.table_name
     AND dq.snapshot_id = load.snapshot_id
)
WHERE validation_rank = 1;

CREATE OR REPLACE TEMPORARY VIEW selected_psgc_validation AS
SELECT *
FROM `04-validation`.silver_dq_results
WHERE table_name = 'silver_psgc_place'
  AND pipeline_run_id = (
      SELECT MAX(run_id) FROM `02-silver`.silver_psgc_place WHERE place_type = 'REGION'
  );

CREATE OR REPLACE TEMPORARY VIEW latest_config_validation AS
SELECT * EXCEPT (validation_rank)
FROM (
    SELECT *, DENSE_RANK() OVER (ORDER BY run_ts DESC, run_id DESC) AS validation_rank
    FROM `04-validation`.silver_config_dq_results
    WHERE layer = 'silver_config'
)
WHERE validation_rank = 1;

SELECT ASSERT_TRUE(
    COUNT(*) = 2
    AND COUNT_IF(status IN ('SUCCESS', 'SKIPPED_IDEMPOTENT')) = 2
    AND COUNT_IF(snapshot_id IS NOT NULL AND TRIM(snapshot_id) <> '') = 2
    AND COUNT_IF(rows_loaded > 0) = 2,
    'STOP: latest MGB or boundary Bronze load is missing, failed, unidentified, or empty'
)
FROM latest_flood_exposure_source_load;

SELECT ASSERT_TRUE(
    COUNT(DISTINCT table_name) = 2
    AND COUNT_IF(action = 'stop' AND status IN ('FAIL', 'ERROR')) = 0,
    'STOP: latest MGB or boundary Bronze validation is missing or unsafe'
)
FROM latest_flood_exposure_bronze_validation;

SELECT ASSERT_TRUE(
    COUNT(*) > 0 AND COUNT_IF(action = 'stop' AND status = 'FAIL') = 0,
    'STOP: selected Silver PSGC validation is missing or unsafe'
)
FROM selected_psgc_validation;

SELECT ASSERT_TRUE(
    COUNT(*) > 0 AND COUNT_IF(action = 'stop' AND status = 'FAIL') = 0,
    'STOP: latest Silver configuration validation is missing or unsafe'
)
FROM latest_config_validation;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT(DISTINCT run_id) = 1
    AND COUNT(DISTINCT source_snapshot_id) = 1
    AND COUNT(DISTINCT source_ingest_run_id) = 1
    AND COUNT(DISTINCT psgc_version) <= 1
    AND COUNT_IF(psgc_code IS NULL OR TRIM(psgc_code) = '') = 0
    AND COUNT(*) = COUNT(DISTINCT psgc_code),
    'STOP: selected official PSGC region set is empty or unsafe'
)
FROM `02-silver`.silver_psgc_place
WHERE place_type = 'REGION';

-- lineage columns only, so this does not parse the large geometry field
SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT(DISTINCT _source_snapshot_id) = 1
    AND COUNT(DISTINCT _ingest_run_id) = 1
    AND MAX(_source_snapshot_id) = (
        SELECT snapshot_id FROM latest_flood_exposure_source_load
        WHERE table_name = 'flood_susceptibility'
    ),
    'STOP: current MGB Bronze rows do not match the validated MGB snapshot'
)
FROM `01-bronze`.flood_susceptibility;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT(DISTINCT _source_snapshot_id) = 1
    AND COUNT(DISTINCT _ingest_run_id) = 1
    AND MAX(_source_snapshot_id) = (
        SELECT snapshot_id FROM latest_flood_exposure_source_load
        WHERE table_name = 'boundaries'
    ),
    'STOP: current region boundaries do not match the validated boundary snapshot'
)
FROM `01-bronze`.boundaries
WHERE LOWER(TRIM(administrative_level)) = 'region';

## 2. Select deterministic source and mapping versions

**What:** declare every rule constant once as a SQL session variable, prove that exactly one active approved MGB mapping version exists, and derive one deterministic flood-exposure run ID.

**Why:** the run ID hashes meaningful contracts only. These are the MGB snapshot, ingest run, and publisher version, then the mapping and mapping source-contract versions. The boundary snapshot, ingest run, and boundary version follow. The PSGC snapshot, run, and version come next, and the area CRS and rule versions close the list. No date, timestamp, UUID, or random value enters logical identity.

**Area CRS decision:** source geometry is GeoJSON in longitude and latitude, which Databricks parses as SRID 4326. Area is never read from SRID 4326 because its units are square degrees. Area is calculated after `ST_TRANSFORM` to EPSG:6933 (WGS 84 / NSIDC EASE-Grid 2.0 Global). It is an equal-area cylindrical projection in metres, so areas stay comparable across the whole Philippines in one CRS. UTM or PRS92 zones are conformal and split the country across zones. The SRID appears once below as a constant session variable, which keeps `ST_TRANSFORM` arguments constant. `area_rule_version` carries it and the clip-dissolve method.

**Version concepts stay separate:** the MGB Bronze snapshot, the optional MGB publisher version from `load_log`, and the mapping table `source_version` (`unversioned-official-code-contract`) are different facts. The publisher version is never inferred from file metadata.

**Tolerance:** `overlap_tolerance_sqkm` is the only numeric topology tolerance. It decides whether cross-level overlap is material and absorbs floating-point noise in overlap differences. It is part of the rule version.

In [0]:
%sql
-- every rule constant is declared once here and referenced by name below
DECLARE OR REPLACE VARIABLE mgb_mapping_source_system STRING DEFAULT 'DENR MGB flood susceptibility';
DECLARE OR REPLACE VARIABLE source_geometry_srid INT DEFAULT 4326;
DECLARE OR REPLACE VARIABLE area_calculation_srid INT DEFAULT 6933;
DECLARE OR REPLACE VARIABLE square_metres_per_sqkm DOUBLE DEFAULT 1000000;
DECLARE OR REPLACE VARIABLE overlap_tolerance_sqkm DOUBLE DEFAULT 0.01;
-- coarse degree-unit screen used only to prove the source crs, never to filter rows
DECLARE OR REPLACE VARIABLE degree_screen_min_longitude DOUBLE DEFAULT 110.0;
DECLARE OR REPLACE VARIABLE degree_screen_max_longitude DOUBLE DEFAULT 130.0;
DECLARE OR REPLACE VARIABLE degree_screen_min_latitude DOUBLE DEFAULT 0.0;
DECLARE OR REPLACE VARIABLE degree_screen_max_latitude DOUBLE DEFAULT 25.0;
DECLARE OR REPLACE VARIABLE area_calculation_crs STRING;
DECLARE OR REPLACE VARIABLE area_rule_version STRING;
DECLARE OR REPLACE VARIABLE transformation_rule_version STRING;

SET VAR area_calculation_crs = CONCAT('EPSG:', area_calculation_srid);
SET VAR area_rule_version = CONCAT(
    'silver-region-flood-exposure-area-v1|clip-dissolve-equal-area|', area_calculation_crs
);
SET VAR transformation_rule_version = CONCAT(
    'silver-region-flood-exposure-v1|overlap-tolerance-sqkm=', overlap_tolerance_sqkm
);

SELECT ASSERT_TRUE(
    COUNT(DISTINCT mapping.mapping_version) = 1
    AND COUNT(DISTINCT mapping.source_version) = 1,
    'STOP: exactly one active approved MGB mapping version and source contract is required'
)
FROM `02-silver`.config_mgb_susceptibility_mapping AS mapping
WHERE mapping.source_system = mgb_mapping_source_system
  AND mapping.approval_status = 'APPROVED'
  AND mapping.is_active = TRUE;

CREATE OR REPLACE TEMPORARY VIEW selected_flood_exposure_versions AS
WITH mgb_context AS (
    SELECT
        MAX(_source_snapshot_id) AS mgb_source_snapshot_id,
        MAX(_ingest_run_id) AS mgb_source_ingest_run_id,
        MAX(_ingested_at) AS mgb_source_load_ts
    FROM `01-bronze`.flood_susceptibility
),
mgb_publisher AS (
    -- optional publisher label recorded by the load that created the selected rows
    SELECT MAX(NULLIF(TRIM(log.source_version), '')) AS mgb_source_version
    FROM `01-bronze`.load_log AS log
    INNER JOIN mgb_context AS context
        ON log.run_id = context.mgb_source_ingest_run_id
    WHERE log.table_name = 'flood_susceptibility'
      AND log.status = 'SUCCESS'
),
mapping_context AS (
    SELECT
        MAX(mapping.mapping_version) AS susceptibility_mapping_version,
        MAX(mapping.source_version) AS mgb_mapping_source_version
    FROM `02-silver`.config_mgb_susceptibility_mapping AS mapping
    WHERE mapping.source_system = mgb_mapping_source_system
      AND mapping.approval_status = 'APPROVED'
      AND mapping.is_active = TRUE
),
boundary_context AS (
    SELECT
        MAX(boundary._source_snapshot_id) AS boundary_source_snapshot_id,
        MAX(boundary._ingest_run_id) AS boundary_source_ingest_run_id,
        MAX(boundary._source_system) AS boundary_source_system,
        MAX(boundary.source_psgc_version) AS boundary_psgc_version,
        MAX(boundary.source_boundary_version) AS boundary_version,
        MAX(boundary._ingested_at) AS boundary_source_load_ts
    FROM `01-bronze`.boundaries AS boundary
    INNER JOIN latest_flood_exposure_source_load AS latest
        ON boundary._source_snapshot_id = latest.snapshot_id
       AND latest.table_name = 'boundaries'
    WHERE LOWER(TRIM(boundary.administrative_level)) = 'region'
),
psgc_context AS (
    SELECT
        MAX(run_id) AS psgc_run_id,
        MAX(source_snapshot_id) AS psgc_source_snapshot_id,
        MAX(source_ingest_run_id) AS psgc_source_ingest_run_id,
        MAX(psgc_version) AS psgc_version,
        MAX(source_load_ts) AS psgc_source_load_ts
    FROM `02-silver`.silver_psgc_place
    WHERE place_type = 'REGION'
)
SELECT
    mgb_context.*, mgb_publisher.*, mapping_context.*, boundary_context.*, psgc_context.*,
    area_calculation_crs AS area_calculation_crs,
    area_rule_version AS area_rule_version,
    transformation_rule_version AS transformation_rule_version,
    SHA2(CONCAT_WS(
        '||',
        mgb_context.mgb_source_snapshot_id, mgb_context.mgb_source_ingest_run_id,
        COALESCE(mgb_publisher.mgb_source_version, 'NO_PUBLISHER_VERSION'),
        mapping_context.susceptibility_mapping_version,
        mapping_context.mgb_mapping_source_version,
        boundary_context.boundary_source_snapshot_id,
        boundary_context.boundary_source_ingest_run_id,
        COALESCE(boundary_context.boundary_version, 'NO_BOUNDARY_VERSION'),
        psgc_context.psgc_source_snapshot_id, psgc_context.psgc_run_id,
        COALESCE(psgc_context.psgc_version, 'NO_PUBLISHER_VERSION'),
        area_calculation_crs, area_rule_version, transformation_rule_version
    ), 256) AS exposure_run_id
FROM mgb_context
CROSS JOIN mgb_publisher
CROSS JOIN mapping_context
CROSS JOIN boundary_context
CROSS JOIN psgc_context;

SELECT ASSERT_TRUE(
    COUNT(*) = 1
    AND COUNT_IF(
        mgb_source_snapshot_id IS NULL OR mgb_source_ingest_run_id IS NULL
        OR susceptibility_mapping_version IS NULL OR mgb_mapping_source_version IS NULL
        OR boundary_source_snapshot_id IS NULL OR boundary_source_ingest_run_id IS NULL
        OR psgc_source_snapshot_id IS NULL OR psgc_run_id IS NULL
    ) = 0,
    'STOP: selected flood-exposure lineage is incomplete'
)
FROM selected_flood_exposure_versions;

SELECT * FROM selected_flood_exposure_versions;

## 3. Profile MGB susceptibility and geometry

**What:** one grouped scan of the selected MGB snapshot by raw code, geometry presence, declared GeoJSON `type`, and encoding evidence.

**Why:** geometry encoding is verified from the data, not assumed from the column name. The quoted-number count shows whether MGB uses the quoted-coordinate encoding observed in the boundary artifact. The `rings` count would reveal Esri JSON instead of GeoJSON.

**Unresolved by design:** blank codes and `No rating` stay visible here as their own raw values. They are not converted to any standard level.

**Next proof:** Step 7 parses and validates every row and reconciles all source rows into mapping and geometry buckets.

In [0]:
%sql
SELECT
    COALESCE(NULLIF(UPPER(TRIM(mgb.flood_susceptibility_code)), ''), '<BLANK>') AS raw_susceptibility_code,
    CASE
        WHEN mgb.geometry_json IS NULL OR TRIM(mgb.geometry_json) = '' THEN 'BLANK_GEOMETRY'
        ELSE 'HAS_GEOMETRY'
    END AS geometry_presence,
    COALESCE(GET_JSON_OBJECT(mgb.geometry_json, '$.type'), '<NO_TYPE_KEY>') AS declared_geojson_type,
    COUNT(*) AS source_rows,
    COUNT_IF(mgb.geometry_json RLIKE '\"-?[0-9]+(\\.[0-9]+)?\"') AS quoted_number_rows,
    COUNT_IF(mgb.geometry_json RLIKE '\"rings\"') AS esri_rings_rows
FROM `01-bronze`.flood_susceptibility AS mgb
CROSS JOIN selected_flood_exposure_versions AS selected
WHERE mgb._source_snapshot_id = selected.mgb_source_snapshot_id
GROUP BY 1, 2, 3
ORDER BY raw_susceptibility_code, geometry_presence, declared_geojson_type;

## 4. Prepare active approved susceptibility mapping

**What:** read only active `APPROVED` rows from the selected mapping version and build the approved level set dynamically.

**Why:** configuration is the only standardization authority. There is no `CASE` mapping of LF, MF, HF, or VHF in this notebook. A raw code may map to at most one level and rank. Each level must have one rank and each rank one level.

**Protects:** the level side of the output grid. If several approved codes ever share one level, `source_susceptibility_code` lists them in sorted order.

**Unresolved by design:** codes absent from approved configuration, including blank and `No rating`, remain `UNMAPPED`. Gold may route them through a governed unknown member later, but Silver never invents one.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW approved_mgb_susceptibility_mapping AS
SELECT
    UPPER(TRIM(mapping.raw_susceptibility_code)) AS raw_susceptibility_code,
    mapping.standardized_level,
    mapping.severity_rank,
    mapping.mapping_rule,
    mapping.mapping_version,
    mapping.source_version
FROM `02-silver`.config_mgb_susceptibility_mapping AS mapping
CROSS JOIN selected_flood_exposure_versions AS selected
WHERE mapping.source_system = mgb_mapping_source_system
  AND mapping.approval_status = 'APPROVED'
  AND mapping.is_active = TRUE
  AND mapping.mapping_version = selected.susceptibility_mapping_version;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT_IF(
        raw_susceptibility_code IS NULL OR raw_susceptibility_code = ''
        OR standardized_level IS NULL OR TRIM(standardized_level) = ''
        OR severity_rank IS NULL OR severity_rank <= 0
    ) = 0
    AND COUNT(*) = COUNT(DISTINCT raw_susceptibility_code),
    'STOP: approved MGB mapping is empty, incomplete, or maps one raw code more than once'
)
FROM approved_mgb_susceptibility_mapping;

SELECT ASSERT_TRUE(
    COUNT(DISTINCT standardized_level) = COUNT(DISTINCT severity_rank)
    AND COUNT(DISTINCT standardized_level)
        = COUNT(DISTINCT NAMED_STRUCT('level', standardized_level, 'rank', severity_rank)),
    'STOP: approved MGB levels and severity ranks are not one-to-one'
)
FROM approved_mgb_susceptibility_mapping;

CREATE OR REPLACE TEMPORARY VIEW approved_susceptibility_levels AS
SELECT
    standardized_level AS flood_susceptibility_level,
    severity_rank,
    CONCAT_WS(',', SORT_ARRAY(COLLECT_SET(raw_susceptibility_code))) AS source_susceptibility_code,
    CONCAT_WS('; ', SORT_ARRAY(COLLECT_SET(mapping_rule))) AS mapping_rule
FROM approved_mgb_susceptibility_mapping
GROUP BY standardized_level, severity_rank;

SELECT * FROM approved_susceptibility_levels ORDER BY severity_rank;

## 5. Prepare official PSGC region set

**What:** select every current official `REGION` row from `silver_psgc_place`.

**Why:** PSGC, not the boundary file, decides which regions exist. NCR stays its own region without an invented province hierarchy. BARMM stays in scope because Table 14 does not depend on DPWH project coverage.

**Protects:** the region side of the output grid. A region without a usable boundary still receives rows, so missing geography stays visible as no data.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW official_flood_exposure_regions AS
SELECT
    psgc_code AS psgc_region_code,
    place_name_raw AS psgc_region_name
FROM `02-silver`.silver_psgc_place
WHERE place_type = 'REGION';

SELECT COUNT(*) AS official_region_count FROM official_flood_exposure_regions;

## 6. Prepare safe region boundary geometry

**What:** parse only `region` rows from the selected boundary snapshot, match their PSGC code exactly to a current official region, and assess each official region once.

**Why:** Table 14 is region-first. It never compares MGB polygons with province, municipality, or barangay shapes. The boundary GeoJSON stores numeric coordinates as quoted strings, so SQL removes quotes only around numeric tokens and bracketed coordinate strings before `TRY_TO_GEOMETRY`. This is the parser already proven in Table 13.

**Boundary statuses:**

- `VALID_REGION_BOUNDARY`: exactly one parseable, nonempty, valid Polygon or MultiPolygon
- `NO_SAFE_REGION_BOUNDARY`: the selected snapshot has no region feature for the current PSGC code
- `AMBIGUOUS_REGION_BOUNDARY`: more than one region feature has the code. Nothing is deduplicated
- `INVALID_REGION_GEOMETRY`: the single feature is unparseable, empty, invalid, or not polygonal. Nothing is repaired

**Unresolved by design:** the selected boundary snapshot predates the current PSGC release. A current region such as Negros Island Region may have no boundary. It keeps its rows with null measures. Nothing splits Regions VI or VII, and no polygon is fabricated.

**Next proof:** SRID, degree-unit envelope, and transformed-CRS checks below stop the run if the boundary CRS contract is wrong.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW region_boundary_candidates AS
SELECT
    boundary.psgc_code AS boundary_psgc_code,
    region.psgc_region_code AS resolved_region_code,
    COUNT(*) OVER (PARTITION BY boundary.psgc_code) AS boundary_feature_count,
    TRY_TO_GEOMETRY(REGEXP_REPLACE(
        REGEXP_REPLACE(
            boundary.geometry_json,
            '\"(\\[[^\"]*\\])\"',
            CONCAT(CHAR(36), '1')
        ),
        '\"(-?[0-9]+(?:\\.[0-9]+)?(?:[Ee][+-]?[0-9]+)?)\"',
        CONCAT(CHAR(36), '1')
    )) AS boundary_geometry
FROM `01-bronze`.boundaries AS boundary
CROSS JOIN selected_flood_exposure_versions AS selected
LEFT JOIN official_flood_exposure_regions AS region
    ON boundary.psgc_code = region.psgc_region_code
WHERE LOWER(TRIM(boundary.administrative_level)) = 'region'
  AND boundary._source_snapshot_id = selected.boundary_source_snapshot_id;

CREATE OR REPLACE TEMPORARY VIEW region_boundary_territory_check AS
WITH province_membership AS (
    SELECT
        region.psgc_region_code,
        boundary.psgc_code AS boundary_province_code,
        current_place.region_code AS current_psgc_region_code
    FROM `01-bronze`.boundaries AS boundary
    CROSS JOIN selected_flood_exposure_versions AS selected
    INNER JOIN official_flood_exposure_regions AS region
        ON SUBSTRING(boundary.psgc_code, 1, 2)
            = SUBSTRING(region.psgc_region_code, 1, 2)
    LEFT JOIN `02-silver`.silver_psgc_place AS current_place
        ON boundary.psgc_code = current_place.psgc_code
    WHERE boundary._source_snapshot_id = selected.boundary_source_snapshot_id
      AND boundary.psgc_code RLIKE '^[0-9]{5}00000$'
      AND boundary.psgc_code NOT RLIKE '^[0-9]{2}00000000$'
)
SELECT
    psgc_region_code,
    COUNT(*) AS province_feature_count,
    COUNT_IF(
        current_psgc_region_code IS NULL
        OR current_psgc_region_code <> psgc_region_code
    ) AS territory_difference_count,
    COALESCE(
        CONCAT_WS(
            '; ',
            SORT_ARRAY(
                COLLECT_SET(
                    CASE
                        WHEN current_psgc_region_code IS NULL
                            OR current_psgc_region_code <> psgc_region_code
                        THEN CONCAT(
                            boundary_province_code,
                            ' now ',
                            COALESCE(current_psgc_region_code, 'unlisted')
                        )
                    END
                )
            )
        ),
        ''
    ) AS territory_difference_details
FROM province_membership
GROUP BY psgc_region_code;

CREATE OR REPLACE TEMPORARY VIEW region_boundary_assessment AS
WITH single_candidate AS (
    SELECT
        resolved_region_code,
        boundary_geometry
    FROM region_boundary_candidates
    WHERE resolved_region_code IS NOT NULL
      AND boundary_feature_count = 1
),
candidate_count AS (
    SELECT
        resolved_region_code,
        COUNT(*) AS boundary_feature_count
    FROM region_boundary_candidates
    WHERE resolved_region_code IS NOT NULL
    GROUP BY resolved_region_code
),
assessed AS (
    SELECT
        region.psgc_region_code,
        COALESCE(candidate_count.boundary_feature_count, 0) AS boundary_feature_count,
        single_candidate.boundary_geometry,
        COALESCE(territory.province_feature_count, 0) AS province_feature_count,
        COALESCE(territory.territory_difference_count, 0)
            AS territory_difference_count,
        territory.territory_difference_details,
        CASE
            WHEN COALESCE(candidate_count.boundary_feature_count, 0) = 0
                THEN 'NO_SAFE_REGION_BOUNDARY'
            WHEN candidate_count.boundary_feature_count > 1
                THEN 'AMBIGUOUS_REGION_BOUNDARY'
            WHEN single_candidate.boundary_geometry IS NULL
                THEN 'INVALID_REGION_GEOMETRY'
            WHEN ST_ISEMPTY(single_candidate.boundary_geometry)
                THEN 'INVALID_REGION_GEOMETRY'
            WHEN REGEXP_REPLACE(
                UPPER(ST_GEOMETRYTYPE(single_candidate.boundary_geometry)),
                '^ST_',
                ''
            ) NOT IN ('POLYGON', 'MULTIPOLYGON')
                THEN 'INVALID_REGION_GEOMETRY'
            WHEN NOT ST_ISVALID(single_candidate.boundary_geometry)
                THEN 'INVALID_REGION_GEOMETRY'
            WHEN COALESCE(territory.territory_difference_count, 0) > 0
                THEN 'BOUNDARY_TERRITORY_MISMATCH'
            ELSE 'VALID_REGION_BOUNDARY'
        END AS region_boundary_status
    FROM official_flood_exposure_regions AS region
    LEFT JOIN candidate_count
        ON region.psgc_region_code = candidate_count.resolved_region_code
    LEFT JOIN single_candidate
        ON region.psgc_region_code = single_candidate.resolved_region_code
    LEFT JOIN region_boundary_territory_check AS territory
        ON region.psgc_region_code = territory.psgc_region_code
),
projected AS (
    SELECT
        assessed.*,
        CASE
            WHEN assessed.region_boundary_status = 'VALID_REGION_BOUNDARY'
                THEN ST_TRANSFORM(
                    assessed.boundary_geometry,
                    area_calculation_srid
                )
        END AS projected_boundary_geometry
    FROM assessed
)
SELECT
    psgc_region_code,
    boundary_feature_count,
    province_feature_count,
    territory_difference_count,
    territory_difference_details,
    region_boundary_status,
    boundary_geometry,
    ST_SRID(projected_boundary_geometry) AS region_area_srid,
    -- area is read only from a safe projected region boundary
    ST_AREA(projected_boundary_geometry)
        / square_metres_per_sqkm AS region_area_sqkm_unrounded
FROM projected;

CREATE OR REPLACE TEMPORARY VIEW safe_region_boundaries AS
SELECT psgc_region_code, boundary_geometry, region_area_sqkm_unrounded
FROM region_boundary_assessment
WHERE region_boundary_status = 'VALID_REGION_BOUNDARY';

SELECT ASSERT_TRUE(
    COUNT_IF(region_boundary_status = 'VALID_REGION_BOUNDARY') > 0,
    'STOP: no official region has a safe boundary, so no exposure can be calculated'
)
FROM region_boundary_assessment;

SELECT ASSERT_TRUE(
    COUNT_IF(
        ST_SRID(boundary_geometry) <> source_geometry_srid
        OR ST_XMIN(boundary_geometry) < degree_screen_min_longitude
        OR ST_XMAX(boundary_geometry) > degree_screen_max_longitude
        OR ST_YMIN(boundary_geometry) < degree_screen_min_latitude
        OR ST_YMAX(boundary_geometry) > degree_screen_max_latitude
    ) = 0,
    'STOP: safe region boundaries are not longitude-latitude geometry in the expected source SRID'
)
FROM safe_region_boundaries;

SELECT ASSERT_TRUE(
    COUNT_IF(
        region_boundary_status = 'VALID_REGION_BOUNDARY'
        AND (
            region_area_srid IS NULL
            OR region_area_srid <> area_calculation_srid
            OR region_area_srid = source_geometry_srid
            OR region_area_sqkm_unrounded IS NULL
            OR region_area_sqkm_unrounded <= 0
        )
    ) = 0,
    'STOP: region area was not calculated in the approved projected equal-area CRS'
)
FROM region_boundary_assessment;

SELECT psgc_region_code, region_boundary_status, boundary_feature_count, region_area_srid,
       ROUND(region_area_sqkm_unrounded, 4) AS region_area_sqkm
FROM region_boundary_assessment
ORDER BY psgc_region_code;

## 7. Parse and validate MGB geometry

**What:** classify every selected MGB row once by mapping status and geometry status, then reconcile all rows before any spatial work.

**Why:** the parser first tries the GeoJSON as published. When that fails, it applies the same targeted quoted-number normalization used for boundaries. It never strips JSON quotes globally. `geometry_parse_method` records which path succeeded, so Step 3 encoding evidence and Step 7 results can be compared.

**Geometry statuses:** `BLANK_GEOMETRY`, `UNPARSEABLE_GEOMETRY`, `EMPTY_GEOMETRY`, `UNSUPPORTED_GEOMETRY_TYPE`, `INVALID_GEOMETRY`, and `USABLE_GEOMETRY`. Only a nonempty, OGC-valid Polygon or MultiPolygon is usable. Invalid geometry is excluded and flagged, never repaired with `ST_BUFFER` or another hidden rule.

**Source accounting (STOP):**

```text
selected MGB rows
= mapped approved + usable geometry
+ mapped approved + unusable geometry
+ unmapped or unrated + usable geometry
+ unmapped or unrated + unusable geometry
```

The selected-row total must also equal the audited `rows_loaded`. The run stops if any mapped usable geometry is not SRID 4326, or if mapped rows have geometry text but none parse. In that case the encoding contract is wrong. Publishing zero exposure would hide the failure.

**Source grain:** no permanent polygon key is invented. Lineage is the source snapshot, ingest run, and versions.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW mgb_source_classified AS
WITH source_rows AS (
    SELECT
        NULLIF(UPPER(TRIM(mgb.flood_susceptibility_code)), '') AS raw_susceptibility_code,
        NULLIF(TRIM(mgb.geometry_json), '') AS geometry_text
    FROM `01-bronze`.flood_susceptibility AS mgb
    CROSS JOIN selected_flood_exposure_versions AS selected
    WHERE mgb._source_snapshot_id = selected.mgb_source_snapshot_id
),
direct_parse AS (
    SELECT source_rows.*, TRY_TO_GEOMETRY(geometry_text) AS published_geometry
    FROM source_rows
),
esri_json_parsed AS (
    SELECT
        direct_parse.*,
        CASE
            WHEN direct_parse.geometry_text IS NOT NULL
                AND GET_JSON_OBJECT(direct_parse.geometry_text, '$.rings') IS NOT NULL
            THEN ST_TRANSFORM(
                ST_SETSRID(
                    CASE
                        WHEN SIZE(FROM_JSON(direct_parse.geometry_text, 'STRUCT<rings: ARRAY<ARRAY<ARRAY<DOUBLE>>>>').rings) <= 1
                        THEN TRY_TO_GEOMETRY(TO_JSON(NAMED_STRUCT(
                            'type', 'Polygon',
                            'coordinates', FROM_JSON(direct_parse.geometry_text, 'STRUCT<rings: ARRAY<ARRAY<ARRAY<DOUBLE>>>>').rings
                        )))
                        ELSE TRY_TO_GEOMETRY(TO_JSON(NAMED_STRUCT(
                            'type', 'MultiPolygon',
                            'coordinates', TRANSFORM(
                                FROM_JSON(direct_parse.geometry_text, 'STRUCT<rings: ARRAY<ARRAY<ARRAY<DOUBLE>>>>').rings,
                                ring -> ARRAY(ring)
                            )
                        )))
                    END,
                    3857
                ),
                4326
            )
            ELSE NULL
        END AS esri_geometry
    FROM direct_parse
),
normalized AS (
    SELECT
        esri_json_parsed.*,
        TRY_TO_GEOMETRY(REGEXP_REPLACE(
            REGEXP_REPLACE(geometry_text, '\"(\\[[^\"]*\\])\"', CONCAT(CHAR(36), '1')),
            '\"(-?[0-9]+(?:\\.[0-9]+)?(?:[Ee][+-]?[0-9]+)?)\"',
            CONCAT(CHAR(36), '1')
        )) AS normalized_geometry
    FROM esri_json_parsed
),
parsed AS (
    SELECT
        normalized.*,
        COALESCE(
            published_geometry,
            normalized_geometry,
            esri_geometry
        ) AS mgb_geometry
    FROM normalized
)
SELECT
    parsed.raw_susceptibility_code,
    parsed.mgb_geometry,
    mapping.standardized_level AS flood_susceptibility_level,
    CASE
        WHEN mapping.raw_susceptibility_code IS NOT NULL THEN 'MAPPED_APPROVED'
        ELSE 'UNMAPPED'
    END AS susceptibility_mapping_status,
    CASE
        WHEN parsed.geometry_text IS NULL THEN 'NOT_PARSED'
        WHEN parsed.published_geometry IS NOT NULL THEN 'GEOJSON_AS_PUBLISHED'
        WHEN parsed.normalized_geometry IS NOT NULL THEN 'GEOJSON_QUOTED_NUMBERS_NORMALIZED'
        WHEN parsed.esri_geometry IS NOT NULL THEN 'ESRI_JSON_CONVERTED'
        ELSE 'PARSE_FAILED'
    END AS geometry_parse_method,
    CASE
        WHEN parsed.geometry_text IS NULL THEN 'BLANK_GEOMETRY'
        WHEN parsed.mgb_geometry IS NULL THEN 'UNPARSEABLE_GEOMETRY'
        WHEN ST_ISEMPTY(parsed.mgb_geometry) THEN 'EMPTY_GEOMETRY'
        WHEN REGEXP_REPLACE(UPPER(ST_GEOMETRYTYPE(parsed.mgb_geometry)), '^ST_', '')
            NOT IN ('POLYGON', 'MULTIPOLYGON') THEN 'UNSUPPORTED_GEOMETRY_TYPE'
        WHEN NOT ST_ISVALID(parsed.mgb_geometry) THEN 'INVALID_GEOMETRY'
        ELSE 'USABLE_GEOMETRY'
    END AS source_geometry_status
FROM parsed
LEFT JOIN approved_mgb_susceptibility_mapping AS mapping
    ON parsed.raw_susceptibility_code = mapping.raw_susceptibility_code;

-- one grouped scan both reports the accounting buckets and enforces the gate
SELECT
    accounting.*,
    ASSERT_TRUE(
        accounting.selected_mgb_rows = accounting.audited_rows_loaded
        AND accounting.selected_mgb_rows
            = accounting.mapped_usable_rows + accounting.mapped_unusable_rows
            + accounting.unmapped_usable_rows + accounting.unmapped_unusable_rows,
        'STOP: MGB source-row accounting does not reconcile'
    ) AS accounting_gate,
    ASSERT_TRUE(
        accounting.usable_wrong_srid_rows = 0,
        'STOP: usable MGB geometry is not in the expected source SRID'
    ) AS source_srid_gate,
    ASSERT_TRUE(
        accounting.mapped_geometry_text_rows = 0 OR accounting.mapped_usable_rows > 0,
        'STOP: no mapped MGB geometry is usable, so the geometry encoding contract is broken'
    ) AS encoding_gate
FROM (
    SELECT
        COUNT(*) AS selected_mgb_rows,
        MAX(load.rows_loaded) AS audited_rows_loaded,
        COUNT_IF(susceptibility_mapping_status = 'MAPPED_APPROVED'
            AND source_geometry_status = 'USABLE_GEOMETRY') AS mapped_usable_rows,
        COUNT_IF(susceptibility_mapping_status = 'MAPPED_APPROVED'
            AND source_geometry_status <> 'USABLE_GEOMETRY') AS mapped_unusable_rows,
        COUNT_IF(susceptibility_mapping_status = 'UNMAPPED'
            AND source_geometry_status = 'USABLE_GEOMETRY') AS unmapped_usable_rows,
        COUNT_IF(susceptibility_mapping_status = 'UNMAPPED'
            AND source_geometry_status <> 'USABLE_GEOMETRY') AS unmapped_unusable_rows,
        COUNT_IF(susceptibility_mapping_status = 'MAPPED_APPROVED'
            AND source_geometry_status <> 'BLANK_GEOMETRY') AS mapped_geometry_text_rows,
        COUNT_IF(source_geometry_status = 'USABLE_GEOMETRY'
            AND ST_SRID(mgb_geometry) <> source_geometry_srid) AS usable_wrong_srid_rows,
        COUNT_IF(source_geometry_status = 'BLANK_GEOMETRY') AS blank_geometry_rows,
        COUNT_IF(source_geometry_status = 'UNPARSEABLE_GEOMETRY') AS unparseable_geometry_rows,
        COUNT_IF(source_geometry_status = 'EMPTY_GEOMETRY') AS empty_geometry_rows,
        COUNT_IF(source_geometry_status = 'UNSUPPORTED_GEOMETRY_TYPE') AS unsupported_geometry_rows,
        COUNT_IF(source_geometry_status = 'INVALID_GEOMETRY') AS invalid_geometry_rows,
        COUNT_IF(geometry_parse_method = 'GEOJSON_AS_PUBLISHED') AS parsed_as_published_rows,
        COUNT_IF(geometry_parse_method = 'GEOJSON_QUOTED_NUMBERS_NORMALIZED') AS parsed_after_normalization_rows,
        COUNT_IF(geometry_parse_method = 'ESRI_JSON_CONVERTED') AS esri_json_converted_rows
    FROM mgb_source_classified
    CROSS JOIN latest_flood_exposure_source_load AS load
    WHERE load.table_name = 'flood_susceptibility'
) AS accounting;

## 8. Perform region × MGB spatial intersection

**What:** join mapped, usable MGB rows to the small broadcast set of safe region boundaries with `ST_INTERSECTS`, then build `ST_INTERSECTION` only for intersecting pairs.

**Why:** this is the one expensive spatial operation in Silver. Its cost is MGB rows × safe regions, not MGB rows × 43,760 boundary shapes and not MGB rows × projects. A polygon that crosses a regional border yields one clipped fragment in each region, so its area is split rather than assigned wholly to one region.

**Area of each fragment** is calculated after projection to the approved equal-area CRS. Fragments with zero area are border touches and do not count as exposure.

**Next proof:** the validator checks that every published area fits inside its region and that dissolve never increases area.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW region_mgb_intersection_fragments AS
WITH clipped AS (
    SELECT /*+ BROADCAST(region) */
        region.psgc_region_code,
        mgb.flood_susceptibility_level,
        ST_INTERSECTION(region.boundary_geometry, mgb.mgb_geometry) AS fragment_geometry
    FROM mgb_source_classified AS mgb
    INNER JOIN safe_region_boundaries AS region
        ON ST_INTERSECTS(region.boundary_geometry, mgb.mgb_geometry)
    WHERE mgb.susceptibility_mapping_status = 'MAPPED_APPROVED'
      AND mgb.source_geometry_status = 'USABLE_GEOMETRY'
),
measured AS (
    SELECT
        clipped.*,
        ST_AREA(ST_TRANSFORM(clipped.fragment_geometry, area_calculation_srid))
            / square_metres_per_sqkm AS fragment_area_sqkm
    FROM clipped
)
SELECT psgc_region_code, flood_susceptibility_level, fragment_geometry, fragment_area_sqkm
FROM measured
WHERE fragment_area_sqkm > 0;

## 9. Dissolve exposure geometry by region and level

**What:** `ST_UNION_AGG` dissolves clipped fragments for each region and level. The same pass also dissolves all approved levels together for each region through `GROUPING SETS`.

**Why:** MGB polygons of the same level may overlap. Summing their areas would double-count, so area is read from the dissolved geometry. The all-level union measures cross-level overlap in the same pass. Without it, the whole intersection would run a second time.

**Counts before dissolve:**

- `source_polygon_count` counts contributing source MGB rows for the region and level.
- `intersection_fragment_count` counts region-by-polygon intersection results.

Each source row meets one region boundary at most once because only single-feature regions are safe. Both counts are therefore equal by construction, and the validator proves it. A polygon crossing two regions contributes to both regional counts. Do not sum these counts as a national unique-polygon total.

**Cross-level overlap:** severity levels are never prioritized. No rule says Very High overrides High. Overlap between levels is measured and published so that Gold does not sum level areas as if they were mutually exclusive.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW region_level_dissolved_exposure AS
SELECT
    psgc_region_code,
    flood_susceptibility_level,
    GROUPING(flood_susceptibility_level) AS is_all_level_union,
    ST_UNION_AGG(fragment_geometry) AS dissolved_geometry,
    COUNT(*) AS intersection_fragment_count,
    SUM(fragment_area_sqkm) AS fragment_area_sum_sqkm
FROM region_mgb_intersection_fragments
GROUP BY GROUPING SETS ((psgc_region_code, flood_susceptibility_level), (psgc_region_code));

## 10. Calculate projected area measures

**What:** project each dissolved geometry to the approved equal-area CRS once, read its area in square kilometres, and derive both overlap measures.

**Formulas:**

```text
susceptible_area_sqkm = ST_AREA(ST_TRANSFORM(dissolved level geometry, approved SRID)) / 1,000,000
within_level_overlap_sqkm = sum of clipped fragment areas - susceptible_area_sqkm
region_cross_level_overlap_sqkm = sum of level areas in region - area of all-level union in region
```

Full precision is kept here. Rounding happens only at publication.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW region_level_exposure_area AS
WITH projected AS (
    SELECT
        dissolved.*,
        ST_TRANSFORM(dissolved.dissolved_geometry, area_calculation_srid) AS projected_geometry
    FROM region_level_dissolved_exposure AS dissolved
),
measured AS (
    SELECT
        psgc_region_code, flood_susceptibility_level, is_all_level_union,
        intersection_fragment_count, fragment_area_sum_sqkm,
        ST_SRID(projected_geometry) AS exposure_area_srid,
        ST_AREA(projected_geometry) / square_metres_per_sqkm AS dissolved_area_sqkm
    FROM projected
),
with_region_union AS (
    SELECT
        measured.*,
        SUM(CASE WHEN is_all_level_union = 0 THEN dissolved_area_sqkm END)
            OVER (PARTITION BY psgc_region_code)
        - MAX(CASE WHEN is_all_level_union = 1 THEN dissolved_area_sqkm END)
            OVER (PARTITION BY psgc_region_code) AS region_cross_level_overlap_sqkm
    FROM measured
)
SELECT
    psgc_region_code,
    flood_susceptibility_level,
    exposure_area_srid,
    dissolved_area_sqkm AS susceptible_area_sqkm_unrounded,
    intersection_fragment_count AS source_polygon_count,
    intersection_fragment_count,
    fragment_area_sum_sqkm - dissolved_area_sqkm AS within_level_overlap_sqkm_unrounded,
    region_cross_level_overlap_sqkm AS region_cross_level_overlap_sqkm_unrounded
FROM with_region_union
WHERE is_all_level_union = 0;

## 11. Build complete region × level output grid

**What:** cross join every official region with every approved level, then left join calculated exposure.

**Zero versus no data:**

| Region boundary | Exposure found | Spatial status | Area, share, and counts |
| --- | --- | --- | --- |
| Safe | Yes | `CALCULATED` | Calculated values |
| Safe | No | `NO_MAPPED_EXPOSURE` | Real zero |
| Missing, ambiguous, or invalid | Not evaluated | `NO_SAFE_REGION_BOUNDARY` | `NULL`, never zero |

**Additivity status:** `ADDITIVE_WITHIN_TOLERANCE` means a safe region's level areas do not materially overlap. `NON_ADDITIVE_CROSS_LEVEL_OVERLAP` means summing its level areas would double-count. `NOT_EVALUATED` applies when no safe boundary exists.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW region_flood_exposure_grid AS
SELECT
    region.psgc_region_code,
    region.psgc_region_name,
    level.source_susceptibility_code,
    level.flood_susceptibility_level,
    level.severity_rank,
    level.mapping_rule,
    boundary.region_boundary_status,
    boundary.boundary_feature_count,
    boundary.province_feature_count,
    boundary.territory_difference_count,
    boundary.territory_difference_details,
    boundary.region_area_sqkm_unrounded,
    boundary.region_boundary_status = 'VALID_REGION_BOUNDARY' AS has_safe_boundary,
    exposure.psgc_region_code IS NOT NULL AS has_exposure,
    exposure.susceptible_area_sqkm_unrounded,
    exposure.source_polygon_count,
    exposure.intersection_fragment_count,
    exposure.within_level_overlap_sqkm_unrounded,
    -- one region value copied to every level row without a second spatial scan
    MAX(exposure.region_cross_level_overlap_sqkm_unrounded)
        OVER (PARTITION BY region.psgc_region_code) AS region_cross_level_overlap_sqkm_unrounded
FROM official_flood_exposure_regions AS region
CROSS JOIN approved_susceptibility_levels AS level
INNER JOIN region_boundary_assessment AS boundary
    ON region.psgc_region_code = boundary.psgc_region_code
LEFT JOIN region_level_exposure_area AS exposure
    ON region.psgc_region_code = exposure.psgc_region_code
   AND level.flood_susceptibility_level = exposure.flood_susceptibility_level;

## 12. Publish `silver_region_flood_exposure`

**What:** publish one deterministic row per grid cell with `CREATE OR REPLACE TABLE`. An exact rerun replaces the same logical rows and keys and never appends duplicates.

**Key:** `region_flood_exposure_key = SHA2(psgc_region_code | flood_susceptibility_level | run_id)`. This is the Silver SHA-256 convention. Gold owns its BIGINT surrogate keys later.

**Types:** areas are `DECIMAL(18,4)` and the share is `DECIMAL(7,4)`. Values are rounded once, here, from full-precision intermediates. Nothing is clamped with `ABS`, `LEAST`, or `GREATEST`. Out-of-range values stay visible to the validator.

**Boundary lineage on no-data rows:** run-level boundary snapshot, ingest, and version columns describe the boundary snapshot that was searched. They are present on every row. `boundary_feature_count` shows whether a feature existed for that region.

**Next:** run `04_validation/07_validation_silver_region_flood_exposure.ipynb`.

In [0]:
%sql
CREATE OR REPLACE TABLE `02-silver`.silver_region_flood_exposure
USING DELTA
COMMENT 'One official region per approved MGB susceptibility level, selected MGB and boundary snapshots, and deterministic flood-exposure run'
AS
SELECT
    SHA2(CONCAT_WS(
        '|', grid.psgc_region_code, grid.flood_susceptibility_level, selected.exposure_run_id
    ), 256) AS region_flood_exposure_key,
    grid.psgc_region_code,
    grid.psgc_region_name,
    grid.source_susceptibility_code,
    grid.flood_susceptibility_level,
    grid.severity_rank,
    CAST(CASE WHEN grid.has_safe_boundary
        THEN ROUND(grid.region_area_sqkm_unrounded, 4) END AS DECIMAL(18, 4)) AS region_area_sqkm,
    CAST(CASE WHEN grid.has_safe_boundary
        THEN ROUND(COALESCE(grid.susceptible_area_sqkm_unrounded, 0), 4) END AS DECIMAL(18, 4))
        AS susceptible_area_sqkm,
    CAST(CASE WHEN grid.has_safe_boundary
        THEN ROUND(
            100 * COALESCE(grid.susceptible_area_sqkm_unrounded, 0)
            / NULLIF(grid.region_area_sqkm_unrounded, 0), 4
        ) END AS DECIMAL(7, 4)) AS share_of_region_area_pct,
    CASE WHEN grid.has_safe_boundary
        THEN COALESCE(grid.source_polygon_count, 0) END AS source_polygon_count,
    CASE WHEN grid.has_safe_boundary
        THEN COALESCE(grid.intersection_fragment_count, 0) END AS intersection_fragment_count,
    CAST(CASE WHEN grid.has_safe_boundary
        THEN ROUND(COALESCE(grid.within_level_overlap_sqkm_unrounded, 0), 4) END AS DECIMAL(18, 4))
        AS within_level_overlap_sqkm,
    CAST(CASE WHEN grid.has_safe_boundary
        THEN ROUND(COALESCE(grid.region_cross_level_overlap_sqkm_unrounded, 0), 4) END AS DECIMAL(18, 4))
        AS region_cross_level_overlap_sqkm,
    CASE
        WHEN NOT grid.has_safe_boundary THEN 'NOT_EVALUATED'
        WHEN COALESCE(grid.region_cross_level_overlap_sqkm_unrounded, 0) > overlap_tolerance_sqkm
            THEN 'NON_ADDITIVE_CROSS_LEVEL_OVERLAP'
        ELSE 'ADDITIVE_WITHIN_TOLERANCE'
    END AS level_area_additivity_status,
    'MAPPED_APPROVED' AS susceptibility_mapping_status,
    grid.region_boundary_status,
    grid.province_feature_count,
    grid.territory_difference_count,
    grid.territory_difference_details,
    CASE
        WHEN NOT grid.has_safe_boundary THEN 'NO_SAFE_REGION_BOUNDARY'
        WHEN grid.has_exposure THEN 'CALCULATED'
        ELSE 'NO_MAPPED_EXPOSURE'
    END AS spatial_match_status,
    CASE
        WHEN COALESCE(selected.boundary_psgc_version, '') = COALESCE(selected.psgc_version, '')
            THEN 'VERSION_ALIGNED'
        ELSE 'VERSION_MISMATCH_VISIBLE'
    END AS boundary_version_status,
    CASE
        WHEN NOT grid.has_safe_boundary THEN 'NOT_EVALUATED'
        WHEN grid.has_exposure THEN 'VALID_SOURCE_GEOMETRY_USED'
        ELSE 'NO_INTERSECTING_VALID_GEOMETRY'
    END AS source_geometry_status,
    CASE grid.region_boundary_status
        WHEN 'NO_SAFE_REGION_BOUNDARY'
            THEN 'SELECTED_BOUNDARY_SNAPSHOT_HAS_NO_REGION_FEATURE_FOR_CURRENT_PSGC_CODE'
        WHEN 'AMBIGUOUS_REGION_BOUNDARY'
            THEN 'MULTIPLE_REGION_BOUNDARY_FEATURES_FOR_CURRENT_PSGC_CODE'
        WHEN 'INVALID_REGION_GEOMETRY'
            THEN 'REGION_BOUNDARY_UNPARSEABLE_EMPTY_INVALID_OR_NOT_POLYGONAL'
        WHEN 'BOUNDARY_TERRITORY_MISMATCH'
            THEN 'BOUNDARY_TERRITORY_DOES_NOT_MATCH_CURRENT_PSGC_MEMBERSHIP'
    END AS exception_reason,
    selected.mgb_source_snapshot_id,
    selected.mgb_source_ingest_run_id,
    selected.mgb_source_version,
    selected.mgb_source_load_ts,
    selected.mgb_mapping_source_version,
    selected.susceptibility_mapping_version,
    grid.mapping_rule,
    selected.boundary_source_snapshot_id,
    selected.boundary_source_ingest_run_id,
    selected.boundary_source_system,
    selected.boundary_psgc_version,
    selected.boundary_version,
    selected.boundary_source_load_ts,
    grid.boundary_feature_count,
    selected.psgc_source_snapshot_id,
    selected.psgc_source_ingest_run_id,
    selected.psgc_version,
    selected.psgc_run_id,
    selected.psgc_source_load_ts,
    selected.area_calculation_crs,
    selected.area_rule_version,
    selected.transformation_rule_version,
    selected.exposure_run_id AS run_id,
    GREATEST(
        selected.mgb_source_load_ts, selected.boundary_source_load_ts, selected.psgc_source_load_ts
    ) AS source_load_ts
FROM region_flood_exposure_grid AS grid
CROSS JOIN selected_flood_exposure_versions AS selected;

# Summary

- **Table:** `02-silver.silver_region_flood_exposure`.
- **Grain:** one official PSGC region × one active approved MGB susceptibility level × the selected MGB snapshot × the selected boundary snapshot × one deterministic flood-exposure run.
- **Primary key:** `region_flood_exposure_key`, a SHA-256 hash of region code, level, and `run_id`.
- **MGB mapping version:** the single active approved `mapping_version` from `config_mgb_susceptibility_mapping`. Its source contract is kept separately in `mgb_mapping_source_version`.
- **Boundary version:** the selected boundary snapshot, ingest run, `boundary_version`, and `boundary_psgc_version` are retained. `boundary_version_status` keeps the PSGC mismatch visible.
- **PSGC version:** the current `silver_psgc_place` snapshot, run, and `psgc_version`.
- **CRS used:** source geometry stays in SRID 4326 for parsing and clipping. Area uses EPSG:6933 after `ST_TRANSFORM` and is never read in square degrees.
- **Area rule version:** `area_rule_version` records the clip-dissolve equal-area method and CRS. `transformation_rule_version` records the overlap tolerance.
- **Missing boundary behavior:** a current region without one safe boundary keeps every level row with `NULL` measures and `NO_SAFE_REGION_BOUNDARY`. It is no data, not zero exposure.
- **Unmapped MGB behavior:** blank, `No rating`, and any unapproved code never contribute to an approved level. They remain in source accounting and are flagged by the validator.
- **Source row accounting:** every selected MGB row lands in one mapping-by-geometry bucket. The total equals the audited Bronze row count.
- **Overlap:** within-level overlap is dissolved before area. Cross-level overlap is measured and published, and no severity precedence is applied.
- **Gold dependency:** Gold `fact_region_flood_exposure` maps `psgc_region_code` and `flood_susceptibility_level` to its keys. It must not repeat spatial processing.
- **Runtime:** requires Databricks execution on a runtime with native spatial SQL (`ST_TRANSFORM` requires Databricks Runtime 17.1 or later). No runtime result is claimed by this notebook text.

In [0]:
%sql
SELECT
    psgc_region_code, psgc_region_name, flood_susceptibility_level, severity_rank,
    region_boundary_status, spatial_match_status, level_area_additivity_status,
    region_area_sqkm, susceptible_area_sqkm, share_of_region_area_pct,
    source_polygon_count, within_level_overlap_sqkm, region_cross_level_overlap_sqkm
FROM `02-silver`.silver_region_flood_exposure
ORDER BY psgc_region_code, severity_rank;